In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib # Used to load your saved model

# --- 1. LOAD YOUR TRAINED MODEL & SCALER ---
# Update these paths to where you saved your model and scaler files
# Example of how to save in your notebook: joblib.dump(log_reg_model, 'loan_model.pkl')
@st.cache_resource
def load_assets():
    try:
        model = joblib.load('loan_model.pkl')
        # scaler = joblib.load('scaler.pkl') # Uncomment if you saved your scaler
        return model #, scaler
    except FileNotFoundError:
        return None #, None

model = load_assets()

# --- 2. APP HEADER & STYLING ---
st.set_page_config(page_title="CreditWise Loan Predictor", page_icon="🏦", layout="centered")
st.title("🏦 CreditWise Loan Eligibility Predictor")
st.markdown("Enter the applicant's details below to predict if their loan will be **Approved** or **Rejected**.")
st.divider()

# --- 3. USER INPUT FORM ---
with st.form("loan_application_form"):
    st.subheader("👤 Applicant Demographics & Education")
    col1, col2 = st.columns(2)
    with col1:
        employment_status = st.selectbox("Employment Status", ["Employed", "Self-Employed", "Unemployed"])
        education_level = st.selectbox("Education Level", ["Graduate", "Not Graduate"])
    with col2:
        property_area = st.selectbox("Property Area", ["Urban", "Semiurban", "Rural"])
        existing_loans = st.number_input("Number of Existing Loans", min_value=0, max_value=10, value=0)

    st.subheader("💰 Financial Background")
    col3, col4 = st.columns(2)
    with col3:
        applicant_income = st.number_input("Applicant Income ($)", min_value=0, value=5000)
        coapplicant_income = st.number_input("Coapplicant Income ($)", min_value=0, value=0)
        savings = st.number_input("Total Savings ($)", min_value=0, value=1000)
    with col4:
        credit_score = st.slider("Credit Score", min_value=300, max_value=850, value=700)
        collateral_value = st.number_input("Collateral Value ($)", min_value=0, value=0)
        dti_ratio = st.number_input("Debt-to-Income (DTI) Ratio", min_value=0.0, max_value=1.0, value=0.3, format="%.2f")

    # Submit button for the form
    submit_button = st.form_submit_button(label="Predict Loan Status")

# --- 4. PREDICTION LOGIC ---
if submit_button:
    if model is None:
        st.error("⚠️ Model file ('loan_model.pkl') not found! Please make sure you saved it from your Jupyter Notebook and placed it in the same folder as app.py.")
    else:
        # 1. Store inputs in a DataFrame exactly how your model expects them
        input_data = pd.DataFrame({
            'Applicant_Income': [applicant_income],
            'Coapplicant_Income': [coapplicant_income],
            'Credit_Score': [credit_score],
            'Existing_Loans': [existing_loans],
            'DTI_Ratio': [dti_ratio],
            'Savings': [savings],
            'Collateral_Value': [collateral_value],
            'Employment_Status': [employment_status],
            'Education_Level': [education_level],
            'Property_Area': [property_area],
            # If you used feature engineering in your notebook, calculate them here!
            # 'DTI_Ratio_sq': [dti_ratio ** 2],
            # 'Credit_Score_sq': [credit_score ** 2]
        })

        # 2. Preprocess the data (Encoding/Scaling)
        # Note: You should apply the EXACT same transformations you did in your notebook.
        # e.g., input_data_scaled = scaler.transform(input_data)
        
        # For this template, we assume your pipeline handles the raw DataFrame:
        try:
            prediction = model.predict(input_data)
            probability = model.predict_proba(input_data)[0][1] * 100

            st.divider()
            
            # 3. Display Results
            if prediction[0] == 1 or prediction[0] == "Yes": # Adjust based on how your target was encoded
                st.success(f"🎉 **Loan Approved!**")
                st.balloons()
            else:
                st.error(f"❌ **Loan Rejected.**")
            
            st.info(f"**Confidence Score:** {probability:.2f}%")
            
        except Exception as e:
            st.error(f"An error occurred during prediction: {e}")
            st.warning("Make sure the columns in `input_data` match exactly what your model was trained on (including order and spelling).")

2026-05-03 11:45:09.546 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 11:45:10.955 
  command:

    streamlit run C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-03 11:45:10.956 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 11:45:10.958 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 11:45:10.959 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 11:45:10.965 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 11:45:10.966 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 11:45:10.967 Thread 'MainThread': mi

In [ ]:
!python -m streamlit run loanfy.py